# 00 — Dân số theo tỉnh theo năm (WorldPop)

Luồng A1 — [PHASE-1-CHECKLIST.md §A1](../../PHASE-1-CHECKLIST.md). Tải raster WorldPop (`data.worldpop.org`, đã kiểm chứng 21/09/2026), cộng dồn về 34 tỉnh bằng zonal sum, xuất `data/interim/population_by_province_year.parquet` cho `build_panel.py` dùng.

**Trước khi chạy toàn bộ:** WorldPop chỉ phủ năm **2000-2020**, mỗi năm ~150-200MB → tải hết 21 năm tốn ~3-4GB — đã đo thật 1 năm ~13 phút ở mạng máy dev, nên cả 21 năm có thể mất vài giờ, không phải vài phút. Cứ để chạy nền, cell tự bỏ qua năm đã tải nếu chạy lại giữa chừng. Cell dưới có biến `YEARS` — đổi thành khoảng nhỏ (vd `range(2018, 2021)`) để chạy thử nhanh trước, rồi đổi lại full range khi chắc chắn pipeline chạy đúng.

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd()
if not (_root / "app").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print("project root:", _root)

In [ ]:
from app.data.ingest_population import (
    WORLDPOP_COVERAGE,
    build_worldpop_panel,
    extend_to_full_range,
)

print(f"WorldPop coverage: {WORLDPOP_COVERAGE}")

In [ ]:
# Đổi range(2000, 2021) -> range(2018, 2021) để test nhanh trước.
YEARS = range(2000, 2021)

worldpop_panel = build_worldpop_panel(years=YEARS)
worldpop_panel.head()

## Mở rộng ra 1994-2025

Năm ngoài phạm vi WorldPop lấy giá trị năm gần nhất đã biết (carry-forward/backward), gắn `population_source='imputed'` — khác `'estimated'` của năm có raster thật. Không được gộp 2 loại này khi báo cáo (xem [docs/01 §8](../../docs/01-chien-luoc-du-lieu.md#8)).

In [ ]:
population_panel = extend_to_full_range(worldpop_panel, range(1994, 2026))
population_panel['population_source'].value_counts()

## Sanity check

Tổng dân số Việt Nam phải nằm trong khoảng hợp lý (~70 triệu năm 1994 → ~100 triệu năm gần đây). Ra số âm, bằng 0, hoặc hàng tỉ là có bug (đơn vị, merge sai, raster đọc nhầm).

In [ ]:
import matplotlib.pyplot as plt

yearly_total = population_panel.groupby('year')['population'].sum()
print(yearly_total.loc[[1994, 2000, 2010, 2020, 2025]])

fig, ax = plt.subplots(figsize=(9, 4))
yearly_total.plot(ax=ax)
ax.set_title('Tổng dân số 34 tỉnh (WorldPop zonal sum + imputed)')
ax.set_ylabel('Dân số')
ax.axvspan(1994, WORLDPOP_COVERAGE[0], alpha=0.15, color='orange', label='imputed (trước WorldPop)')
ax.axvspan(WORLDPOP_COVERAGE[1], 2025, alpha=0.15, color='orange')
ax.legend()
plt.show()

In [ ]:
assert 60_000_000 < yearly_total.min() < 110_000_000, "Tổng dân số ra ngoài khoảng hợp lý — kiểm tra lại trước khi lưu"
print('Sanity check pass.')

## Lưu ra `data/interim/`

In [ ]:
from pathlib import Path

interim_dir = _root / 'data' / 'interim'
interim_dir.mkdir(parents=True, exist_ok=True)
out_path = interim_dir / 'population_by_province_year.parquet'
population_panel.to_parquet(out_path, index=False)
print(f'Đã lưu {len(population_panel)} dòng vào {out_path}')

**Tiếp theo:** chạy `01_ingest_oni.ipynb` và `02_ingest_era5.ipynb`, sau đó `python -m app.data.build_panel` từ `ai-service/` để ghép panel v0.2.0.